In [1]:
# importing all necessary libraries
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from tqdm.notebook import tqdm as tqdm
import plotly.graph_objs as go #visualization library
import random
import os
import time
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import gc
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
import multiprocessing as mp
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline
from datetime import datetime
from torch.utils.data import Dataset, DataLoader
from pandas.tseries import offsets
from pandas.tseries.frequencies import to_offset
from typing import List

In [2]:
root_path='ETT-small'
data_path='ETTh2.csv'
df_raw = pd.read_csv(os.path.join(root_path,data_path))

In [3]:
df_raw.head()

,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,41.130001,12.481,36.535999,9.355,4.424,1.311,38.661999
1,2016-07-01 01:00:00,37.528000,10.136,33.936001,7.532,4.435,1.215,37.124001
2,2016-07-01 02:00:00,37.946999,11.309,35.330002,9.007,2.100,0.000,36.465000
3,2016-07-01 03:00:00,38.952000,11.895,35.543999,9.436,3.380,1.215,33.608501
4,2016-07-01 04:00:00,38.113998,11.476,35.410000,9.623,2.036,0.000,31.850500


In [4]:
df_raw['date'] = pd.to_datetime(df_raw.date)

In [5]:
df_raw['month']   = df_raw['date'].dt.month
df_raw['day']     = df_raw['date'].dt.day
df_raw['weekday'] = df_raw['date'].dt.weekday
df_raw['hour']    = df_raw['date'].dt.hour
# Preprocessing pipeline in Pandas
df_raw['hour_norm'] = (df_raw['hour'] / 23.0) - 0.5
df_raw['weekday_norm'] = (df_raw['weekday'] / 6.0) - 0.5
df_raw['month_norm'] = (df_raw['month'] / 12.0) - 0.5

In [ ]:
df_raw.head()

In [6]:
import numpy as np
import pandas as pd

# --- Configuration ---
TARGET = "OT"
FEATURES = ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'month_norm', 'weekday_norm', 'hour_norm', TARGET]
TIMESTEPS = 24
PREDICTION_HORIZON = 48

# --- Data Preparation ---
# Filter and cast to a clean NumPy array for maximum computational efficiency
df_data = df_raw[FEATURES].copy()
data_arr = df_data.values
target_idx = df_data.columns.get_loc(TARGET)

total_len = len(df_data)

# --- Vectorized Sliding Window Extraction ---
# Replaces the slow nested loop + .shift() architecture with direct NumPy slicing
X_data = np.array([
    data_arr[i - TIMESTEPS + 1 : i + 1] 
    for i in range(TIMESTEPS, total_len - PREDICTION_HORIZON)
])

y_data = np.array([
    data_arr[i + 1 : i + 1 + PREDICTION_HORIZON, target_idx] 
    for i in range(TIMESTEPS, total_len - PREDICTION_HORIZON)
])[..., np.newaxis]  # Expands dimensions to (Samples, Horizon, 1)

# --- Verification ---
print(f"X shape: {X_data.shape}")  # Expected: (Samples, TIMESTEPS, Features)
print(f"y shape: {y_data.shape}")  # Expected: (Samples, PREDICTION_HORIZON, 1)

X shape: (17348, 24, 10)
y shape: (17348, 48, 1)


# Extract the last observation of each feature

In [7]:
nb_pats, seq, n_features = X_data.shape
timeseries_last_obs_data = []
for i in range(nb_pats):
    Index_Last=(~np.isnan(X_data[i,:,:])).cumsum(0).argmax(0)
    Last_Indices = np.reshape(Index_Last,(1,n_features))
    Last_Values = np.take_along_axis(X_data[i,:,:], Last_Indices, axis = 0)
    timeseries_last_obs_data.append(np.repeat(Last_Values, seq, axis=0))
last_obs_data=np.stack(timeseries_last_obs_data)

# The frequency observation of each feature -----> np.isfinite

In [8]:
# isnan
freq_list= []
nb_pats, seq, n_features = X_data.shape
for i in range(nb_pats):
    data_samples=  np.expand_dims(X_data[i,:,:], axis=0)
    nan_counts = np.sum(np.isnan(data_samples), axis=(0, 1))
    freq_list.append(np.repeat(np.expand_dims(nan_counts, axis=0), seq, axis=0))
freqs = np.stack(freq_list)

In [9]:
freqs, freqs.shape

(array([[[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        ...,
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],

In [10]:
import math
import re
import itertools  

import datetime
from collections import namedtuple, defaultdict
from tqdm import tqdm
from itertools import groupby
from operator import itemgetter
from functools import partial
from itertools import combinations

In [11]:
def forward_imputation(data):
    store_index = []
    for index in  range(len(data)):
        try: 
            if len(data)>1:
                store_index.append([data[index], data[index+1]])
            else:
                store_index.append(data[index])
        except:
            pass
    return store_index
def _forward_with_last_measured(data, hours_data=48):
    cols_data = data.columns.to_list()
    times_hours_data = np.arange(0,hours_data,1)
    notna_cols_indexes, nan_cols_indexes  = defaultdict(list),defaultdict(list)
    indexes_last_indicator = defaultdict(list)
    for col in cols_data:
        notna_cols_indexes[col].append(list(data.loc[pd.notna(data[col]), :].index))
        nan_cols_indexes[col].append(list(data.loc[pd.isna(data[col]), :].index))
        indexes_last_indicator[col].append([data[col].notna()[::-1].idxmax(),
                                      data[col].isna()[::-1].idxmax()])
    notna_cols_indexes_ = {key: list(itertools.chain.from_iterable(value)) 
                                 for key , value in notna_cols_indexes.items() if list(itertools.chain.from_iterable(value))}
    notna_cols_indexes_ ={ key:value  for key , value in notna_cols_indexes_.items() if len(value)!=len(times_hours_data)}

    nan_cols_indexes = {key: list(itertools.chain.from_iterable(value)) 
                                for key , value in nan_cols_indexes.items()}
    nan_cols_indexes_ ={key:value  for key , value in nan_cols_indexes.items() 
                                if len(value)!=len(times_hours_data)}
    nan_cols_indexes_ ={key:value  for key , value in nan_cols_indexes_.items() if value}
    
    indexes_last_indicator = {key: list(itertools.chain.from_iterable(value)) 
                                    for key , value in indexes_last_indicator.items()}
    matrix_indexes_notna = {key:forward_imputation(value) for key, value in notna_cols_indexes_.items()}
    matrix_notna_with_last = {key:list(itertools.chain.from_iterable([matrix_indexes_notna[key], [indexes_last_indicator[key]]])) 
                                 for key in indexes_last_indicator if key in matrix_indexes_notna }
    final_matrix_indexes = defaultdict(list)
    for key, vals in matrix_notna_with_last.items():
        for val in vals:
            if isinstance(val, list):
                final_matrix_indexes[key].append(val)
    matrix_range_indexes_cols = {key:[final_matrix_indexes[key], nan_cols_indexes_[key]] 
                                 for key in nan_cols_indexes_ if key in final_matrix_indexes}
    
    range_indexes_cols_imputed = {}
    for key, value in matrix_range_indexes_cols.items():
        range_values = [[notna_ind,nan] for nan in value[1] 
                      for notna_ind in value[0] if notna_ind[0]<=nan<=notna_ind[1]]
        range_indexes_cols_imputed[key]=range_values
    return range_indexes_cols_imputed

def forward_with_last_measured_value_with_time_elasped_interval(data_copy, mask_data, hrs_used=48):
    resultats = _forward_with_last_measured(data_copy, hours_data=hrs_used)
    data =data_copy.copy()
    mask_forward =mask_data.copy()
    for key , values in resultats.items():
        for k, group in groupby(values, lambda x:x[0]):
            vals_ = list(itertools.chain.from_iterable(list(group)))
            vals_ = np.array([val for val in vals_ if not isinstance(val, list)])
            if k[1]<hrs_used-1:  
                for index in np.arange(k[0]+1, k[1]):
                    data.at[index, key]= data._get_value(k[0], key)
                    mask_forward.at[index, key]= index-k[0]
            else:
                for index in np.arange(k[0]+1, k[1]+1):
                    if pd.isnull(data._get_value(index, key)):
                        #print(index, key, data[key])
                        data.at[index, key]= data._get_value(k[0], key)
                        mask_forward.at[index, key]= index-k[0]
                    else:
                        pass
    return data, mask_forward  

In [12]:
x_features = FEATURES
print(x_features)

['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'month_norm', 'weekday_norm', 'hour_norm', 'OT']


In [13]:
TARGET

'OT'

In [14]:
TIMESTEPS

24

In [15]:
timedelta, features_inputs= [],[]
for x_data,freq in tqdm(zip(X_data, freqs), total=len(y_data),
                            desc='Iterating over samples'):
    masked_timeseries = np.where((pd.isnull(x_data)), np.nan, 0)
    features_data = pd.DataFrame(x_data, columns=x_features)
    masked_episode_timeseries = pd.DataFrame(masked_timeseries, columns=x_features)
    data_for_imp,time_elasped_timeseries=forward_with_last_measured_value_with_time_elasped_interval(features_data,
                                                                                                     masked_episode_timeseries,
                                                                                                     hrs_used=TIMESTEPS)
    timedelta.append(time_elasped_timeseries.values)
    features_inputs.append(masked_episode_timeseries.values)

Iterating over samples: 100%|██████████| 17348/17348 [00:44<00:00, 386.29it/s]


In [ ]:
delta_time_data=np.stack(timedelta)
mask_data=np.stack(features_inputs)

In [ ]:
delta_time_data

In [ ]:
delta_time_data[np.isnan(delta_time_data)],  delta_time_data[np.isinf(delta_time_data)]

In [ ]:
data_path, data_path.split(".")[0]

In [ ]:
dataset_name =f"{data_path.split('.')[0]}_{TIMESTEPS}_timesteps_AHEAD_horizon_{PREDICTION_HORIZON}_RUNS".upper()
dataset_name

In [ ]:
dn=f"TIMESERIES/{dataset_name}"
if not os.path.exists(dn):
    os.makedirs(dn)
dn, dataset_name

In [ ]:
delta_time_data[np.isnan(delta_time_data)] = 999
delta_time_data[np.isinf(delta_time_data)] = 999

In [ ]:
TARGETS =np.squeeze(y_data,axis=-1)
TARGETS

# Cross validation on timeseries data

In [ ]:
# ---------------------------------------------------------------------------
# Manual Min-Max Helpers (Global Temporal) with Clipping
# ---------------------------------------------------------------------------

def get_minmax_params_3d(x: np.ndarray):
    """Computes min and max for each feature across all N and T."""
    x_min = np.nanmin(x, axis=(0, 1), keepdims=True)
    x_max = np.nanmax(x, axis=(0, 1), keepdims=True)
    
    # Avoid division by zero
    x_max[x_max == x_min] += 1e-8
    return x_min, x_max

def apply_minmax_3d(x: np.ndarray, x_min: np.ndarray, x_max: np.ndarray) -> np.ndarray:
    """Applies scaling and clips values to the [0, 1] range."""
    normed = (x - x_min) / (x_max - x_min)
    return np.clip(normed, 0, 1) # Clipping added here

def get_minmax_params_2d(x: np.ndarray):
    """Computes min/max for 2D arrays (N, D)."""
    x_min = np.nanmin(x, axis=0, keepdims=True)
    x_max = np.nanmax(x, axis=0, keepdims=True)
    x_max[x_max == x_min] += 1e-8
    return x_min, x_max

def apply_minmax_2d(x: np.ndarray, x_min: np.ndarray, x_max: np.ndarray) -> np.ndarray:
    """Applies scaling and clips values to the [0, 1] range."""
    normed = (x - x_min) / (x_max - x_min)
    return normed # Clipping added here

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

def get_loaders(x, t, last, freq, y, batch_size, shuffle=False):
    """Wraps NumPy arrays into PyTorch DataLoaders."""
    dataset = TensorDataset(
        torch.tensor(x, dtype=torch.float32),
        torch.tensor(t, dtype=torch.float32),
        torch.tensor(last, dtype=torch.float32),
        torch.tensor(freq, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=4)

def split_data_fixed_proportions(data, time_data, last_data, features_freqs, target, batch_size=32):
    # --- Step 1: Separate 20% of the total data as the Test Set ---
    # (Remaining 80% is train_val)
    indices = np.arange(len(data))
    train_val_idx, test_idx = train_test_split(
        indices, 
        test_size=0.20, 
        random_state=10,
        shuffle=False # Set to False if you want to keep chronological order
    )

    # --- Step 2: From the 80% training block, use 20% for Validation ---
    # (The remaining 80% of this block becomes the final training data)
    tr_idx, val_idx = train_test_split(
        train_val_idx, 
        test_size=0.20, 
        random_state=10,
        shuffle=False
    )

    # Slice the actual data using indices
    x_train, x_val, x_test = data[tr_idx], data[val_idx], data[test_idx]
    y_train, y_val, y_test = target[tr_idx], target[val_idx], target[test_idx]
    
    # Auxiliary features
    t_train, t_val, t_test = time_data[tr_idx], time_data[val_idx], time_data[test_idx]
    l_train, l_val, l_test = last_data[tr_idx], last_data[val_idx], last_data[test_idx]
    f_train, f_val, f_test = features_freqs[tr_idx], features_freqs[val_idx], features_freqs[test_idx]

    # --- Step 3: Fit Scalers ONLY on the final training set ---
    x_min, x_max = get_minmax_params_3d(x_train)
    l_min, l_max = get_minmax_params_3d(l_train) 
    y_min, y_max = get_minmax_params_2d(y_train)

    # --- Step 4: Transform all sets using Training Parameters ---
    x_train_s, x_val_s, x_test_s = [apply_minmax_3d(d, x_min, x_max) for d in [x_train, x_val, x_test]]
    l_train_s, l_val_s, l_test_s = [apply_minmax_3d(d, l_min, l_max) for d in [l_train, l_val, l_test]]
    y_train_s, y_val_s, y_test_s = [apply_minmax_2d(d, y_min, y_max) for d in [y_train, y_val, y_test]]

    # --- Step 5: Create DataLoaders ---
    train_loader = get_loaders(x_train_s, t_train, l_train_s, f_train, y_train_s, batch_size, shuffle=True)
    val_loader   = get_loaders(x_val_s, t_val, l_val_s, f_val, y_val_s, batch_size, shuffle=False)
    test_loader  = get_loaders(x_test_s, t_test, l_test_s, f_test, y_test_s, batch_size, shuffle=False)

    scaler_params = {'x': (x_min, x_max), 'last': (l_min, l_max), 'y': (y_min, y_max)}

    print(f"Final Split Proportions:")
    print(f"Train: {len(tr_idx)} samples ({len(tr_idx)/len(data)*100:.1f}%)")
    print(f"Val:   {len(val_idx)} samples ({len(val_idx)/len(data)*100:.1f}%)")
    print(f"Test:  {len(test_idx)} samples ({len(test_idx)/len(data)*100:.1f}%)")

    return train_loader, val_loader, test_loader, scaler_params

In [ ]:
train_loader, val_loader, test_loader, scaler_params = split_data_fixed_proportions(X_data,   
                                                                                   delta_time_data,last_obs_data, 
                                                                                   freqs,TARGETS, batch_size=64)
training_iterations = [(train_loader, val_loader, scaler_params)] * 5
testing_iterations = [(test_loader)] * 5

In [ ]:
seq_length = X_data.shape[1]
input_dim = X_data.shape[-1]
hidden_dim, output_dim  = 64, TARGETS.shape[-1]
taskname=dataset_name
task=f"{os.path.join(dn, f'{taskname}'.upper())}"
if not os.path.exists(task):
    os.makedirs(task)
np.savez(os.path.join(task, f"train_test_data.npz"), 
            folds_data_test= testing_iterations,
            folds_data_train_valid= training_iterations,)
        
np.savez(os.path.join(task, f"data_max_min.npz"), 
         seq_length=seq_length,input_dim=input_dim,
output_dim=TARGETS.shape[-1])
input_dim, seq_length, output_dim, task

In [ ]:
dn